# 01 – Getting to Know the Data

## Problem
Are property prices in Mexico driven more by **property size** or by **location**?

## Data
Three raw CSV exports of property listings in `data/raw/`, each in a different format:

| File | Price column | Location columns |
|---|---|---|
| `mexico-real-estate-1.csv` | `price_usd` (text, e.g. `"$67,965.56"`) | `state`, `lat`, `lon` |
| `mexico-real-estate-2.csv` | `price_mxn` (pesos) | `state`, `lat`, `lon` |
| `mexico-real-estate-3.csv` | `price_usd` | `place_with_parent_names`, `lat-lon` (combined) |

## Goal of this notebook
Inspect each file, clean it, combine all three into one tidy dataset, and save it to `data/processed/`.

In [2]:
import pandas as pd
print(pd.__version__)

3.0.6


In [3]:
from pathlib import Path

print(Path.cwd())

/workspaces/Mexico_real_estate_analysis/notebooks


In [4]:
# Paths are relative to the notebooks/ folder, where this notebook runs.
# ".." means "go up one level" to the project root.
path_1 = Path("../data/raw/mexico-real-estate-1.csv")
path_2 = Path("../data/raw/mexico-real-estate-2.csv")
path_3 = Path("../data/raw/mexico-real-estate-3.csv")

# Check that Python can actually find each file
for path in [path_1, path_2, path_3]:
    print(path, "->", path.exists())

../data/raw/mexico-real-estate-1.csv -> True
../data/raw/mexico-real-estate-2.csv -> True
../data/raw/mexico-real-estate-3.csv -> True


In [5]:
pd.read_csv(path_1).head()

,property_type,state,lat,lon,area_m2,price_usd
0,house,Estado de México,19.560181,-99.233528,150.0,"$67,965.56"
1,house,Nuevo León,25.688436,-100.198807,186.0,"$63,223.78"
2,apartment,Guerrero,16.767704,-99.764383,82.0,"$84,298.37"
3,apartment,Guerrero,16.829782,-99.911012,150.0,"$94,308.80"
4,house,Veracruz de Ignacio de la Llave,NaN,NaN,175.0,"$94,835.67"


In [6]:
pd.read_csv(path_1).shape

(700, 6)

In [7]:
pd.read_csv(path_1).info()

<class 'pandas.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   property_type  700 non-null    str    
 1   state          700 non-null    str    
 2   lat            583 non-null    float64
 3   lon            583 non-null    float64
 4   area_m2        700 non-null    float64
 5   price_usd      700 non-null    str    
dtypes: float64(3), str(3)
memory usage: 32.9 KB


In [8]:
pd.read_csv(path_1).dtypes

property_type        str
state                str
lat              float64
lon              float64
area_m2          float64
price_usd            str
dtype: object

In [9]:
pd.read_csv(path_1).isnull().sum()

property_type      0
state              0
lat              117
lon              117
area_m2            0
price_usd          0
dtype: int64

## File 1 – Inspection findings

| Check | Result |
|---|---|
| Size | 700 rows × 6 columns |
| Missing values | `lat` and `lon`: 117 each (~17%); all other columns complete |
| Type issue | `price_usd` stored as text (`"$67,965.56"`), not numeric |
| Coverage | Listings span many states, not just Mexico City |

### Cleaning plan
1. Drop rows with missing coordinates (no reliable way to fill them in).
2. Strip `$` and `,` from `price_usd`, then convert to float.
3. Verify: expect 583 rows left, `price_usd` as `float64`, zero missing values.

In [10]:
(
    pd.read_csv(path_1)
    .dropna()
    .shape
)

(583, 6)

In [11]:
(
    pd.read_csv(path_1)
    .dropna()
    .assign(
        price_usd=lambda x: x["price_usd"]
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .astype(float)
    )
    .dtypes
)

property_type        str
state                str
lat              float64
lon              float64
area_m2          float64
price_usd        float64
dtype: object

In [12]:
df1 = (
    pd.read_csv(path_1)
    .dropna()
    .assign(
        price_usd=lambda x: x["price_usd"]
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .astype(float)
    )
)

# Verify the cleaning did what the plan said
print("Shape:", df1.shape)
print("price_usd dtype:", df1["price_usd"].dtype)
print("Missing values:", df1.isnull().sum().sum())

Shape: (583, 6)
price_usd dtype: float64
Missing values: 0


## File 2 – Inspection

In [13]:
pd.read_csv(path_2).head()

,property_type,state,lat,lon,area_m2,price_mxn
0,apartment,Nuevo León,25.721081,-100.345581,72.0,1300000.0
1,apartment,Puebla,NaN,NaN,190.0,2500000.0
2,house,Morelos,23.634501,-102.552788,360.0,5300000.0
3,house,Morelos,NaN,NaN,76.0,820000.0
4,house,Puebla,NaN,NaN,200.0,1100000.0


In [14]:
df2_raw = pd.read_csv(path_2)

print("Shape:", df2_raw.shape)
print()
df2_raw.info()
print()
print(df2_raw.isnull().sum())

Shape: (700, 6)

<class 'pandas.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   property_type  700 non-null    str    
 1   state          700 non-null    str    
 2   lat            571 non-null    float64
 3   lon            571 non-null    float64
 4   area_m2        700 non-null    float64
 5   price_mxn      700 non-null    float64
dtypes: float64(4), str(2)
memory usage: 32.9 KB

property_type      0
state              0
lat              129
lon              129
area_m2            0
price_mxn          0
dtype: int64


In [15]:
# Assumption: fixed exchange rate (see the File 2 findings above)
MXN_PER_USD = 19

(
    pd.read_csv(path_2)
    .assign(price_usd=lambda x: x["price_mxn"].div(MXN_PER_USD))
    .head()
)

,property_type,state,lat,lon,area_m2,price_mxn,price_usd
0,apartment,Nuevo León,25.721081,-100.345581,72.0,1300000.0,68421.052632
1,apartment,Puebla,NaN,NaN,190.0,2500000.0,131578.947368
2,house,Morelos,23.634501,-102.552788,360.0,5300000.0,278947.368421
3,house,Morelos,NaN,NaN,76.0,820000.0,43157.894737
4,house,Puebla,NaN,NaN,200.0,1100000.0,57894.736842


In [16]:
df2 = (
    pd.read_csv(path_2)
    .assign(price_usd=lambda x: x["price_mxn"].div(MXN_PER_USD))
    .drop(columns=["price_mxn"])
    .dropna()
)

# Verify against the plan
print("Shape:", df2.shape)
print("Same columns as df1 (same order):", df2.columns.tolist() == df1.columns.tolist())
print("price_usd dtype:", df2["price_usd"].dtype)
print("Missing values:", df2.isnull().sum().sum())

Shape: (571, 6)
Same columns as df1 (same order): True
price_usd dtype: float64
Missing values: 0


## File 3 – Inspection

In [17]:
df3_raw = pd.read_csv(path_3)

print("Shape:", df3_raw.shape)
print()
df3_raw.info()
print()
print(df3_raw.isnull().sum())
print()
display(df3_raw.head())

Shape: (700, 5)

<class 'pandas.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   property_type            700 non-null    str    
 1   place_with_parent_names  700 non-null    str    
 2   lat-lon                  582 non-null    str    
 3   area_m2                  700 non-null    float64
 4   price_usd                700 non-null    float64
dtypes: float64(2), str(3)
memory usage: 27.5 KB

property_type                0
place_with_parent_names      0
lat-lon                    118
area_m2                      0
price_usd                    0
dtype: int64



,property_type,place_with_parent_names,lat-lon,area_m2,price_usd
0,apartment,|México|Distrito Federal|Gustavo A. Madero|Acu...,"19.52589,-99.151703",71.0,48550.59
1,house,|México|Estado de México|Toluca|Metepec|,"19.2640539,-99.5727534",233.0,168636.73
2,house,|México|Estado de México|Toluca|Toluca de Lerd...,"19.268629,-99.671722",300.0,86932.69
3,house,|México|Morelos|Temixco|Burgos Bugambilias|,NaN,275.0,263432.41
4,apartment,|México|Veracruz de Ignacio de la Llave|Veracruz|,"19.511938,-96.871956",84.0,68508.67


In [18]:
display(df3_raw[["lat-lon", "place_with_parent_names"]].head(10))

,lat-lon,place_with_parent_names
0,"19.52589,-99.151703",|México|Distrito Federal|Gustavo A. Madero|Acu...
1,"19.2640539,-99.5727534",|México|Estado de México|Toluca|Metepec|
2,"19.268629,-99.671722",|México|Estado de México|Toluca|Toluca de Lerd...
3,NaN,|México|Morelos|Temixco|Burgos Bugambilias|
4,"19.511938,-96.871956",|México|Veracruz de Ignacio de la Llave|Veracruz|
5,"20.689157,-103.366728",|México|Jalisco|Guadalajara|
6,NaN,|México|Sinaloa|Mazatlán|
7,NaN,|México|Tamaulipas|Tampico|
8,NaN,|México|Distrito Federal|Miguel Hidalgo|
9,"21.0441317,-89.6216334",|México|Yucatán|Mérida|


## File 3 – Inspection findings

| Check | Result |
|---|---|
| Size | 700 rows × 5 columns |
| Missing values | `lat-lon`: 118 (~17%); all other columns complete |
| Type issue | None for price: `price_usd` is already numeric |
| Structure issue | `lat-lon` packs two variables into one text cell (`"19.52589,-99.151703"`). `place_with_parent_names` packs a place hierarchy into one text cell (`"|México|Estado de México|Toluca|..."`) |

### Cleaning plan
1. Drop rows with missing `lat-lon`.
2. Split `lat-lon` on `,` into numeric `lat` and `lon`.
3. Extract `state` from the place path (split on `|`, take index 2).
4. Drop `lat-lon` and `place_with_parent_names`.
5. Verify: expect 582 rows, columns match `df1`, zero missing values.

In [19]:
(
    pd.read_csv(path_3)
    .dropna()
    ["lat-lon"]
    .str.split(",", expand=True)
    .head()
)

,0,1
0,19.52589,-99.151703
1,19.2640539,-99.5727534
2,19.268629,-99.671722
4,19.511938,-96.871956
5,20.689157,-103.366728


In [20]:
(
    pd.read_csv(path_3)
    .dropna()
    .assign(
        lat=lambda x: x["lat-lon"].str.split(",").str[0].astype(float),
        lon=lambda x: x["lat-lon"].str.split(",").str[1].astype(float),
    )
    [["lat-lon", "lat", "lon"]]
    .head()
)

,lat-lon,lat,lon
0,"19.52589,-99.151703",19.525890,-99.151703
1,"19.2640539,-99.5727534",19.264054,-99.572753
2,"19.268629,-99.671722",19.268629,-99.671722
4,"19.511938,-96.871956",19.511938,-96.871956
5,"20.689157,-103.366728",20.689157,-103.366728


In [21]:
(
    pd.read_csv(path_3)
    .assign(state=lambda x: x["place_with_parent_names"].str.split("|").str[2])
    [["place_with_parent_names", "state"]]
    .head(10)
)

,place_with_parent_names,state
0,|México|Distrito Federal|Gustavo A. Madero|Acu...,Distrito Federal
1,|México|Estado de México|Toluca|Metepec|,Estado de México
2,|México|Estado de México|Toluca|Toluca de Lerd...,Estado de México
3,|México|Morelos|Temixco|Burgos Bugambilias|,Morelos
4,|México|Veracruz de Ignacio de la Llave|Veracruz|,Veracruz de Ignacio de la Llave
5,|México|Jalisco|Guadalajara|,Jalisco
6,|México|Sinaloa|Mazatlán|,Sinaloa
7,|México|Tamaulipas|Tampico|,Tamaulipas
8,|México|Distrito Federal|Miguel Hidalgo|,Distrito Federal
9,|México|Yucatán|Mérida|,Yucatán


In [22]:
df3 = (
    pd.read_csv(path_3)
    .dropna()
    .assign(
        lat=lambda x: x["lat-lon"].str.split(",").str[0].astype(float),
        lon=lambda x: x["lat-lon"].str.split(",").str[1].astype(float),
        state=lambda x: x["place_with_parent_names"].str.split("|").str[2],
    )
    [["property_type", "state", "lat", "lon", "area_m2", "price_usd"]]
)

# Verify against the plan
print("Shape:", df3.shape)
print("Same columns as df1 (same order):", df3.columns.tolist() == df1.columns.tolist())
print("Same types as df1:", df3.dtypes.equals(df1.dtypes))
print("Missing values:", df3.isnull().sum().sum())

Shape: (582, 6)
Same columns as df1 (same order): True
Same types as df1: False
Missing values: 0


In [23]:
pd.DataFrame({"df1": df1.dtypes, "df3": df3.dtypes})

,df1,df3
property_type,str,str
state,str,object
lat,float64,float64
lon,float64,float64
area_m2,float64,float64
price_usd,float64,float64


In [24]:
df3 = (
    pd.read_csv(path_3)
    .dropna()
    .assign(
        lat=lambda x: x["lat-lon"].str.split(",").str[0].astype(float),
        lon=lambda x: x["lat-lon"].str.split(",").str[1].astype(float),
        state=lambda x: x["place_with_parent_names"].str.split("|").str[2].astype(str),
    )
    [["property_type", "state", "lat", "lon", "area_m2", "price_usd"]]
)

# Verify against the plan
print("Shape:", df3.shape)
print("Same columns as df1 (same order):", df3.columns.tolist() == df1.columns.tolist())
print("Same types as df1:", df3.dtypes.equals(df1.dtypes))
print("Missing values:", df3.isnull().sum().sum())

Shape: (582, 6)
Same columns as df1 (same order): True
Same types as df1: True
Missing values: 0
